# Job Application Mail Sender (branded HTML)

Picks up jobs that `job_finder_agent.ipynb` added to the Google Sheet, and for every
row that has a **Contact Email** but no **Mail Sent At** yet:

1. Researches the company on the internet (DuckDuckGo)
2. Finds the company website and extracts its **color palette**
3. Writes a **designed HTML email** with Gemini: a soft gradient background built
   from light tints of the company's own colors (always light enough for dark
   text), with the key details — skills, years of experience, metrics —
   **highlighted** in the body
4. Sends it via Gmail SMTP: plain-text fallback + HTML + `resume.pdf` attached
5. Stamps the row's **Mail Sent At** column so it is never emailed twice

In dry-run mode every mail is saved to `previews/<company>.html` — open in a browser
to see exactly what will land in the inbox.

## One-time setup — Gmail App Password

1. Enable 2-Step Verification: https://myaccount.google.com/security
2. Create an App Password: https://myaccount.google.com/apppasswords (app: *Mail*)
3. Add to the project `.env`:
   ```
   GMAIL_ADDRESS=you@gmail.com
   GMAIL_APP_PASSWORD=abcd efgh ijkl mnop
   ```

Everything else (Sheet, service account, Gemini keys) reuses the setup from
`job_finder_agent.ipynb`.

In [216]:
import os
import sys
from pathlib import Path

# make ../ (updatedlangchain/) importable for the shared `common` package
sys.path.append(str(Path.cwd().parent))

from common.api_key_service import get_google_service

google_service = get_google_service()

SHEET_ID = os.getenv('SHEET_ID')
GMAIL_ADDRESS = os.getenv('GMAIL_ADDRESS')
GMAIL_APP_PASSWORD = (os.getenv('GMAIL_APP_PASSWORD') or '').replace(' ', '')

print('Google keys found:', [name for name, _ in google_service.keys])
print('Sheet ID loaded:', bool(SHEET_ID))
print('Gmail configured:', bool(GMAIL_ADDRESS and GMAIL_APP_PASSWORD))

Google keys found: ['GOOGLE_API_KEY_1']
Sheet ID loaded: True
Gmail configured: True


## Connect to the sheet

Same sheet as the finder. Adds a **Mail Sent At** column (K) if it is not there yet.

In [217]:
import gspread

COL_JOB_TITLE = 2      # B
COL_COMPANY = 3        # C
COL_LOCATION = 4       # D
COL_APPLY_LINK = 9     # I
COL_CONTACT_EMAIL = 10 # J
COL_MAIL_SENT = 11     # K

gc = gspread.service_account(filename='service_account.json')
worksheet = gc.open_by_key(SHEET_ID).sheet1

if worksheet.cell(1, COL_MAIL_SENT).value != 'Mail Sent At':
    worksheet.update_cell(1, COL_MAIL_SENT, 'Mail Sent At')

print('Connected to sheet:', worksheet.spreadsheet.title)

Connected to sheet: Copy of Job Openings


## Load resume

In [218]:
from pypdf import PdfReader

RESUME_PATH = Path.cwd().parent / 'resume.pdf'

reader = PdfReader(RESUME_PATH)
resume_text = '\n'.join(page.extract_text() for page in reader.pages)

print(resume_text[:300])

Nikhilesh Ramoliya  
+ 9 1  -  8 4 6 9 1 7 5 2 9 9 n i k h i l e s h r a m o l i y a @ g m a i l . c o m L i n k e d I n
S K I L L S
L a n g u a g e s :  J a v a S c r i p t ,  T y p e S c r i p t ,  P y t h o n
F r o n t e n d :  R e a c t . j s ,  N e x t . j s ,  R e d u x ,  R e a c t  Q u e r y


## Find rows that still need a mail

A row qualifies when **Contact Email** is filled and **Mail Sent At** is empty.

In [219]:
import re

EMAIL_RE = re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}')


def get_pending_jobs() -> list[dict]:
    """Sheet rows with a contact email and no Mail Sent At stamp."""
    pending = []
    for row_idx, row in enumerate(worksheet.get_all_values()[1:], start=2):
        row += [''] * (COL_MAIL_SENT - len(row))  # pad short rows
        emails = EMAIL_RE.findall(row[COL_CONTACT_EMAIL - 1])
        if not emails or row[COL_MAIL_SENT - 1].strip():
            continue
        pending.append({
            'row': row_idx,
            'title': row[COL_JOB_TITLE - 1],
            'company': row[COL_COMPANY - 1],
            'location': row[COL_LOCATION - 1],
            'apply_link': row[COL_APPLY_LINK - 1],
            'emails': emails,
        })
    return pending


pending = get_pending_jobs()
print(f'{len(pending)} jobs pending application mail')
for job in pending:
    print(f"  row {job['row']}: {job['title']} @ {job['company']} -> {job['emails']}")

1 jobs pending application mail
  row 40: Fullstack Engineer (MERN) — Contract, Remote @ LaunchDarkly -> ['nikhilesh.r@lanatussystems.com']


## Research the company

A few targeted DuckDuckGo searches per company; raw snippets are handed to Gemini.
Quoted company name keeps results on-topic (an unquoted name like *Lanatus systems*
returns watermelon taxonomy).

In [220]:
from ddgs import DDGS


def research_company(company: str) -> str:
    """Collect search snippets about the company: what it does, stack, news."""
    queries = [
        f'"{company}" company about',
        f'"{company}" products services technology stack',
        f'"{company}" news 2026',
    ]
    snippets = []
    with DDGS() as ddgs:
        for query in queries:
            try:
                for hit in ddgs.text(query, max_results=4):
                    snippets.append(f"- {hit['title']}: {hit['body']}")
            except Exception as e:
                print(f'  search failed for {query!r}: {e}')
    # dedupe, keep order
    seen = set()
    unique = [s for s in snippets if not (s in seen or seen.add(s))]
    return '\n'.join(unique[:12]) or '(no search results found)'

## Company branding: website → verify it's really them → color palette + light tints

1. **Website** — DuckDuckGo for the official site, skipping job boards / socials
2. **Verify** — a found site only counts if the company name actually shows up in
   its domain, `<title>` or site metadata (`og:site_name` etc.). Search sometimes
   returns a similarly-named but wrong company — without this check we'd put the
   wrong brand's colors on the mail. Several candidate sites are tried; if none
   verifies, the mail falls back to a **default gray/white** look.
3. **Palette** — `theme-color` meta + hex colors from inline styles and the first
   few stylesheets, ranked by frequency, near-white/near-black dropped
4. **Tints** — each brand color blended 88% towards white, pre-computed in code so
   the gradient background is guaranteed light enough for black text (no trusting
   the LLM with contrast)

In [221]:
import requests
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

UA = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'}
SKIP_DOMAINS = (
    'linkedin.', 'indeed.', 'glassdoor.', 'naukri.', 'wikipedia.', 'facebook.',
    'instagram.', 'twitter.', 'x.com', 'youtube.', 'crunchbase.', 'ambitionbox.',
    'zaubacorp.', 'justdial.', 'g2.com', 'clutch.co',
)

# generic words that don't identify a company — ignored when matching name to site
COMPANY_SUFFIXES = {
    'private', 'limited', 'pvt', 'ltd', 'llc', 'llp', 'inc', 'co', 'corp',
    'corporation', 'company', 'group', 'global', 'india', 'technologies',
    'technology', 'tech', 'solutions', 'solution', 'services', 'service',
    'systems', 'system', 'software', 'infotech', 'labs', 'lab', 'studio',
    'studios', 'digital', 'consulting', 'consultancy',
}


def _squash(text: str) -> str:
    return re.sub(r'[^a-z0-9]', '', (text or '').lower())


def _company_tokens(company: str) -> list[str]:
    tokens = re.findall(r'[a-z0-9]+', company.lower())
    core = [t for t in tokens if t not in COMPANY_SUFFIXES]
    return core or tokens  # name made only of generic words -> keep them all


def verify_company_site(company: str, soup: BeautifulSoup, url: str) -> bool:
    """Does this site really belong to the company?

    The company name (minus generic suffixes like Pvt/Ltd/Technologies) must
    appear in the domain, <title>, og:site_name/og:title or application-name.
    """
    tokens = _company_tokens(company)
    name = _squash(''.join(tokens))
    domain = _squash(urlparse(url).netloc.removeprefix('www.'))

    meta_bits = [soup.title.string if soup.title and soup.title.string else '']
    for prop in ('og:site_name', 'og:title'):
        if (m := soup.find('meta', property=prop)) and m.get('content'):
            meta_bits.append(m['content'])
    if (m := soup.find('meta', attrs={'name': 'application-name'})) and m.get('content'):
        meta_bits.append(m['content'])
    meta = _squash(' '.join(meta_bits))

    if name and (name in domain or name in meta):
        return True
    # one distinctive token in the domain (e.g. 'lanatus' in lanatussystems.com)
    if any(len(t) >= 5 and t in domain for t in tokens):
        return True
    # or at least two name tokens somewhere in domain/title/meta
    return sum(1 for t in tokens if len(t) >= 3 and (t in domain or t in meta)) >= 2


def find_company_sites(company: str) -> list[str]:
    """Candidate official-site URLs from search (job boards/socials skipped)."""
    with DDGS() as ddgs:
        try:
            hits = list(ddgs.text(f'"{company}" official website', max_results=8))
        except Exception as e:
            print(f'  site search failed: {e}')
            return []
    sites = []
    for hit in hits:
        url = hit.get('href', '')
        parsed = urlparse(url)
        host = parsed.netloc.lower()
        if host and not any(d in host for d in SKIP_DOMAINS):
            site = f'{parsed.scheme}://{host}'
            if site not in sites:
                sites.append(site)
    return sites[:5]


def fetch(url: str) -> requests.Response | None:
    try:
        resp = requests.get(url, headers=UA, timeout=15)
        resp.raise_for_status()
        return resp
    except Exception:
        return None

In [222]:
HEX_RE = re.compile(r'#([0-9a-fA-F]{6}|[0-9a-fA-F]{3})\b')


def _hex_to_rgb(h: str) -> tuple[int, int, int]:
    h = h.lstrip('#')
    if len(h) == 3:
        h = ''.join(c * 2 for c in h)
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))


def _is_usable_color(h: str) -> bool:
    """Drop near-white, near-black and washed-out grays."""
    r, g, b = _hex_to_rgb(h)
    avg = (r + g + b) / 3
    return 25 < avg < 235 and (max(r, g, b) - min(r, g, b)) > 20


def lighten(h: str, factor: float = 0.88) -> str:
    """Blend a hex color towards white; 0.88 keeps black text readable on it."""
    r, g, b = (round(c + (255 - c) * factor) for c in _hex_to_rgb(h))
    return f'#{r:02x}{g:02x}{b:02x}'


def extract_palette(soup: BeautifulSoup, html: str, base_url: str) -> list[str]:
    """Brand colors, most-used first."""
    counts: dict[str, int] = {}

    def tally(text: str, weight: int = 1):
        for m in HEX_RE.findall(text):
            h = '#' + (''.join(c * 2 for c in m) if len(m) == 3 else m).lower()
            if _is_usable_color(h):
                counts[h] = counts.get(h, 0) + weight

    # theme-color meta is the site's own declared brand color — weight it heavily
    for meta in soup.find_all('meta', attrs={'name': 'theme-color'}):
        tally(meta.get('content', ''), weight=50)

    tally(html)

    for link in soup.find_all('link', rel='stylesheet')[:3]:
        href = link.get('href')
        if href and (resp := fetch(urljoin(base_url, href))):
            tally(resp.text)

    return sorted(counts, key=counts.get, reverse=True)[:6]


def get_branding(company: str) -> dict:
    """{'site': url|None, 'verified': bool, 'palette': ['#...'], 'tints': ['#...']}

    Colors come only from a site that passes verify_company_site. No verified
    site -> empty palette/tints -> the mail uses the default gray/white look.
    """
    branding = {'site': None, 'verified': False, 'palette': [], 'tints': []}
    for site in find_company_sites(company):
        resp = fetch(site)
        if not resp:
            continue
        soup = BeautifulSoup(resp.text, 'html.parser')
        if not verify_company_site(company, soup, site):
            print(f'  site does not match company, colors skipped: {site}')
            continue
        branding['site'] = site
        branding['verified'] = True
        branding['palette'] = extract_palette(soup, resp.text, site)
        branding['tints'] = [lighten(c) for c in branding['palette'][:3]]
        break
    return branding


if pending:
    b = get_branding(pending[0]['company'])
    print('site:', b['site'], ' verified:', b['verified'])
    print('palette:', b['palette'])
    print('tints:', b['tints'])

site: https://launchdarkly.com  verified: True
palette: ['#405bff', '#ebff38', '#3dd6f5', '#a34fde', '#ff9d29', '#e75fa4']
tints: ['#e8ebff', '#fdffe7', '#e8fafe']


## Write the designed HTML mail

One structured Gemini call per job: resume + research + brand palette + safe light
tints in, `{subject, html_body, plain_body}` out. Design: soft gradient page
background from the pre-computed tints (always light enough for dark text), company
name in a colored header band, and the key details — skills, years of experience,
metrics — highlighted in the body.

In [223]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

GEMINI_MODEL = 'gemini-3.1-flash-lite'

# used when no verified company site -> neutral gray/white design
DEFAULT_PALETTE = ['#374151', '#6b7280']
DEFAULT_TINTS = ['#f6f7f8', '#ececec']


class ApplicationMail(BaseModel):
    """A customized, branded job application email."""
    subject: str = Field(description='Email subject line')
    html_body: str = Field(description='Complete HTML email document')
    plain_body: str = Field(description='Plain-text fallback with the same content')


def write_mail(job: dict, research: str, branding: dict) -> ApplicationMail:
    if branding['verified'] and branding['palette']:
        palette = ', '.join(branding['palette'])
        tints = ', '.join(branding['tints'])
        color_origin = "the company's own website"
    else:
        palette = ', '.join(DEFAULT_PALETTE)
        tints = ', '.join(DEFAULT_TINTS)
        color_origin = (
            'a default neutral set (no verified company site — do NOT invent '
            'brand colors, keep the design gray/white)'
        )

    prompt = f"""You are writing a job application email on behalf of the candidate below,
as a polished, branded HTML email.

JOB:
- Title: {job['title']}
- Company: {job['company']}
- Location: {job['location'] or 'Remote'}
- Company website: {branding['site'] or 'unknown'}

COLORS (from {color_origin}):
- ACCENT COLORS (hex, strongest first): {palette}
- SAFE LIGHT TINTS (pre-computed, dark text is always readable on these — use
  ONLY these for any background behind body text): {tints}

COMPANY RESEARCH (web search snippets — use only facts clearly about this company;
ignore anything that looks like a different company with a similar name):
{research}

CANDIDATE RESUME:
{resume_text}

CONTENT (same in html_body and plain_body):
- Greeting to the hiring team
- 1st paragraph: why this company specifically — use real researched facts; if the
  research is empty or off-topic, focus on the role instead of inventing facts
- 2nd paragraph: match the candidate's strongest relevant skills, projects and
  measurable achievements from the resume to this role and company
- Closing: mention the resume is attached, call to action, candidate's name,
  phone and email from the resume
- Under 250 words. Professional, specific, no generic filler, no placeholders.
- subject: candidate name + the exact job title

HTML DESIGN RULES (email clients are primitive — follow strictly):
- Complete document, table-based layout, ALL styles inline (style="...");
  no <style> blocks, no external CSS, no JavaScript, no web fonts, NO images
- Single centered white content card, max-width 600px, rounded corners
- PAGE BACKGROUND: a soft diagonal gradient built ONLY from the safe light tints,
  e.g. style="background-color: <first tint>; background: linear-gradient(135deg,
  <tint1>, <tint2>);" — the background-color first is the fallback for clients
  that drop gradients (Outlook). Never use the raw accent colors as a text background.
- Header band at the top of the card: the company name (text only) and the role,
  using the strongest accent color — either as band background with white text,
  or as colored text on white. High contrast either way.
- HIGHLIGHT the key details in the body text — the candidate's core skills
  (e.g. React, Node.js), years of experience, and standout metrics (e.g. cost
  reductions, performance gains): wrap each in <strong> with the strongest accent
  color as text color, or a subtle padded span using a light tint background.
  5–8 highlights total, no more — they must draw the eye, not wallpaper the mail.
- Accent-colored thin divider or left-border detail welcome; elegant, not flashy.
- Body text dark (#222–#333) on white. System font stack: Arial, Helvetica, sans-serif.
- Use ONLY the colors given above — never make up other colors.
- Footer line with the candidate's contact details in muted gray.
- Keep total HTML under 25 KB."""

    def do_invoke(key: str) -> ApplicationMail:
        model = ChatGoogleGenerativeAI(model=GEMINI_MODEL, google_api_key=key)
        return model.with_structured_output(ApplicationMail).invoke(prompt)

    return google_service.call(do_invoke)

## Send via Gmail SMTP

Multipart structure: plain-text fallback + HTML alternative, plus `resume.pdf`
as a regular attachment.

In [224]:
import smtplib
from email.message import EmailMessage


def build_message(to_addrs: list[str], mail: ApplicationMail) -> EmailMessage:
    msg = EmailMessage()
    msg['From'] = GMAIL_ADDRESS
    msg['To'] = ', '.join(to_addrs)
    msg['Subject'] = mail.subject

    # mark as important / high priority (red ! in Outlook, priority hint elsewhere)
    msg['X-Priority'] = '1'
    msg['X-MSMail-Priority'] = 'High'
    msg["Priority"] = "urgent"
    msg["Importance"] = "high"

    msg.set_content(mail.plain_body)
    msg.add_alternative(mail.html_body, subtype='html')

    msg.add_attachment(
        RESUME_PATH.read_bytes(),
        maintype='application',
        subtype='pdf',
        filename='resume.pdf',
    )
    return msg


def send_mail(msg: EmailMessage) -> None:
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        smtp.send_message(msg)

## Run

**`DRY_RUN = True`** (default) — researches, brands and writes every mail, saves it
to `previews/<company>.html`, but sends nothing and stamps nothing. Open the
previews in a browser, then flip to `False` and re-run to actually send.
Already-stamped rows are always skipped, so re-running is safe.

In [225]:
from datetime import datetime, timezone

DRY_RUN = False
MAX_MAILS_PER_RUN = 10  # safety cap

PREVIEW_DIR = Path('previews')
PREVIEW_DIR.mkdir(exist_ok=True)


def save_preview(job: dict, mail: ApplicationMail) -> Path:
    safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', job['company']).strip('_') or f"row{job['row']}"
    path = PREVIEW_DIR / f'{safe_name}.html'
    path.write_text(mail.html_body, encoding='utf-8')
    return path


pending = get_pending_jobs()
print(f'{len(pending)} pending, sending up to {MAX_MAILS_PER_RUN} (dry run: {DRY_RUN})\n')

sent = 0
for job in pending[:MAX_MAILS_PER_RUN]:
    print(f"=== {job['title']} @ {job['company']} -> {job['emails']} ===")

    research = research_company(job['company'])
    branding = get_branding(job['company'])
    print(f"  site: {branding['site']}  verified: {branding['verified']}  "
          f"palette: {branding['palette'][:3]}  tints: {branding['tints']}")

    mail = write_mail(job, research, branding)
    print('  Subject:', mail.subject)

    preview = save_preview(job, mail)
    print(f'  preview: {preview}')

    if DRY_RUN:
        print()
        continue

    try:
        send_mail(build_message(job['emails'], mail))
    except Exception as e:
        print(f'  SEND FAILED: {e}\n')
        continue

    stamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')
    worksheet.update_cell(job['row'], COL_MAIL_SENT, stamp)
    sent += 1
    print(f'  sent + row {job["row"]} stamped\n')

print('Done. Mails sent:', sent)

1 pending, sending up to 10 (dry run: False)

=== Fullstack Engineer (MERN) — Contract, Remote @ LaunchDarkly -> ['nikhilesh.r@lanatussystems.com'] ===
  site: https://launchdarkly.com  verified: True  palette: ['#405bff', '#ebff38', '#3dd6f5']  tints: ['#e8ebff', '#fdffe7', '#e8fafe']
  Subject: Nikhilesh Ramoliya - Fullstack Engineer (MERN)
  preview: previews/LaunchDarkly.html
  sent + row 40 stamped

Done. Mails sent: 1
